In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
# Expected file: ../data/co2_emissions.csv

data_path = Path("../data/co2_emissions.csv")
if not data_path.exists():
    data_path = Path("data/co2_emissions.csv")
if not data_path.exists():
    data_path = Path("co2_emissions.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "Could not find co2_emissions.csv. Place it in the project's data/ "
        "folder (one level above this notebook, or alongside the notebook in data/)."
    )

df = pd.read_csv(data_path)

# Standardise column names in case the CSV uses slightly different formatting.
df.columns = [column.strip() for column in df.columns]

required_columns = {"Country", "Year", "CO2_Mt", "Region"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

# Accept common variants of the per-capita column name.
per_capita_candidates = [
    "CO2_per_Capita", "CO2_Per_Capita", "CO2 per Capita",
    "CO2_per_capita", "Per_Capita_CO2"
]
per_capita_column = next(
    (column for column in per_capita_candidates if column in df.columns),
    None
)
if per_capita_column is None:
    raise ValueError(
        "Could not find the CO2 per-capita column. Expected one of: "
        + ", ".join(per_capita_candidates)
    )

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["CO2_Mt"] = pd.to_numeric(df["CO2_Mt"], errors="coerce")
df[per_capita_column] = pd.to_numeric(df[per_capita_column], errors="coerce")
df = df.dropna(subset=["Country", "Year", "CO2_Mt", per_capita_column, "Region"])
df["Year"] = df["Year"].astype(int)

print(
    f"Loaded: {len(df):,} rows | Countries: {df['Country'].nunique()} | "
    f"Years: {df['Year'].min()}-{df['Year'].max()}"
)
print("Per-capita column:", per_capita_column)
display(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
Per-capita column: CO2_per_capita


,Country,Region,Year,CO2_Mt,CO2_per_capita
0,United States,North America,2000,5857.6,1.32
1,United States,North America,2001,5724.0,1.26
2,United States,North America,2002,5652.8,1.11
3,United States,North America,2003,5592.8,1.29
4,United States,North America,2004,5743.2,1.12


# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country/countries **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country/countries and its story

In [3]:
# Task 1: CO2 emissions per capita for all countries
# Highlight India; show every other country in light grey for context.

highlight_country = "India"

# Calculate one value per country and year in case the source contains duplicates.
plot_data = (
    df.groupby(["Country", "Year"], as_index=False)
      .agg(CO2_per_capita=(per_capita_column, "mean"))
)

if highlight_country not in plot_data["Country"].unique():
    # Use the country with the highest latest-year per-capita value if India is absent.
    latest_year = plot_data["Year"].max()
    highlight_country = (
        plot_data[plot_data["Year"] == latest_year]
        .sort_values("CO2_per_capita", ascending=False)
        .iloc[0]["Country"]
    )

fig1 = go.Figure()

# Draw all non-highlighted countries first as thin, light-grey lines.
for country, country_data in plot_data.groupby("Country"):
    if country == highlight_country:
        continue
    country_data = country_data.sort_values("Year")
    fig1.add_trace(
        go.Scatter(
            x=country_data["Year"],
            y=country_data["CO2_per_capita"],
            mode="lines",
            line=dict(color="#EEEEEE", width=1),
            hovertemplate=(
                f"<b>{country}</b><br>Year: %{{x}}"
                "<br>CO2 per capita: %{y:.2f} tonnes<extra></extra>"
            ),
            showlegend=False
        )
    )

# Add the highlighted country as a stronger coloured line.
highlight_data = (
    plot_data[plot_data["Country"] == highlight_country]
    .sort_values("Year")
)
fig1.add_trace(
    go.Scatter(
        x=highlight_data["Year"],
        y=highlight_data["CO2_per_capita"],
        mode="lines",
        line=dict(color="#D62728", width=3),
        name=highlight_country,
        showlegend=False,
        hovertemplate=(
            f"<b>{highlight_country}</b><br>Year: %{{x}}"
            "<br>CO2 per capita: %{y:.2f} tonnes<extra></extra>"
        )
    )
)

# Label the highlighted line directly at its final data point.
last_point = highlight_data.iloc[-1]
fig1.add_annotation(
    x=last_point["Year"],
    y=last_point["CO2_per_capita"],
    text=highlight_country,
    showarrow=False,
    xshift=35,
    font=dict(size=12, color="#D62728")
)

fig1.update_layout(
    title=dict(
        text=(
            f"{highlight_country}'s CO2 emissions per person nearly tripled from 2000 to 2022"
            "<br><sup>All other countries shown in grey for comparison</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO2 emissions per capita (tonnes per person)",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=65, r=100, t=90, b=60),
    hovermode="x"
)
fig1.update_xaxes(showgrid=False, showline=True, linecolor="#CCCCCC")
fig1.update_yaxes(showgrid=True, gridcolor="#F0F0F0", zeroline=False)
fig1.show()

print(
    f"Highlighted country: {highlight_country}. "
    f"Latest available value: {last_point['CO2_per_capita']:.2f} tonnes per person "
    f"({int(last_point['Year'])})."
)

Highlighted country: India. Latest available value: 3.36 tonnes per person (2022).


### Task 1 — Interpretation

India's CO2 emissions per person increased from **1.21 tonnes in 2000** to **3.36 tonnes in 2022** in this dataset. That is an increase of **2.15 tonnes per person**, or approximately **2.8 times** the 2000 level. The grey lines provide context by showing the per-capita trends of all other countries in the dataset. This is a per-person measure, so it should not be confused with a country's total emissions.


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

### Task 2 — Interpretation

India's total CO2 emissions increased from **901.6 million tonnes in 2000** to **2,661.1 million tonnes in 2022** in this dataset. However, India is not the largest emitter among the Asian countries shown: China has the highest total emissions in 2022 (**12,409.5 million tonnes**). Japan's total emissions declined over the same period, while the other highlighted Asian countries increased. Total emissions reflect both population size and emissions per person, so this chart tells a different story from Task 1.


In [4]:
# Task 2: Total CO2 emissions over time for Asian countries
# Highlight India; show all other Asian countries in light grey.

asia_data = df[df["Region"].astype(str).str.strip().str.lower() == "asia"].copy()

if asia_data.empty:
    raise ValueError(
        "No rows were found for Region='Asia'. Check the Region values in the dataset."
    )

# Aggregate emissions by country and year.
asia_plot_data = (
    asia_data.groupby(["Country", "Year"], as_index=False)
             .agg(CO2_Mt=("CO2_Mt", "sum"))
)

highlight_asian_country = "India"
if highlight_asian_country not in asia_plot_data["Country"].unique():
    # If India is not present, highlight the Asian country with the highest
    # total emissions in the latest available year.
    latest_year_asia = asia_plot_data["Year"].max()
    highlight_asian_country = (
        asia_plot_data[asia_plot_data["Year"] == latest_year_asia]
        .sort_values("CO2_Mt", ascending=False)
        .iloc[0]["Country"]
    )

fig2 = go.Figure()

# Draw every other Asian country in grey.
for country, country_data in asia_plot_data.groupby("Country"):
    if country == highlight_asian_country:
        continue
    country_data = country_data.sort_values("Year")
    fig2.add_trace(
        go.Scatter(
            x=country_data["Year"],
            y=country_data["CO2_Mt"],
            mode="lines",
            line=dict(color="#EEEEEE", width=1),
            showlegend=False,
            hovertemplate=(
                f"<b>{country}</b><br>Year: %{{x}}"
                "<br>CO2 emissions: %{y:,.1f} Mt<extra></extra>"
            )
        )
    )

# Highlight India (or the fallback country) in colour.
highlight_asia_data = (
    asia_plot_data[asia_plot_data["Country"] == highlight_asian_country]
    .sort_values("Year")
)
fig2.add_trace(
    go.Scatter(
        x=highlight_asia_data["Year"],
        y=highlight_asia_data["CO2_Mt"],
        mode="lines",
        line=dict(color="#D62728", width=3),
        showlegend=False,
        hovertemplate=(
            f"<b>{highlight_asian_country}</b><br>Year: %{{x}}"
            "<br>CO2 emissions: %{y:,.1f} Mt<extra></extra>"
        )
    )
)

# Direct label at the end of the highlighted line.
last_asia_point = highlight_asia_data.iloc[-1]
fig2.add_annotation(
    x=last_asia_point["Year"],
    y=last_asia_point["CO2_Mt"],
    text=highlight_asian_country,
    showarrow=False,
    xshift=35,
    font=dict(size=12, color="#D62728")
)

fig2.update_layout(
    title=dict(
        text=(
            f"{highlight_asian_country}'s total CO2 emissions more than doubled from 2000 to 2022"
            "<br><sup>Total annual CO2 emissions; other Asian countries shown in grey</sup>"
        ),
        x=0.02,
        xanchor="left"
    ),
    xaxis_title="Year",
    yaxis_title="CO2 emissions (million tonnes)",
    template="plotly_white",
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=65, r=100, t=90, b=60),
    hovermode="x"
)
fig2.update_xaxes(showgrid=False, showline=True, linecolor="#CCCCCC")
fig2.update_yaxes(showgrid=True, gridcolor="#F0F0F0", zeroline=False)
fig2.show()

print(
    f"Highlighted Asian country: {highlight_asian_country}. "
    f"Latest available total emissions: {last_asia_point['CO2_Mt']:,.1f} Mt "
    f"({int(last_asia_point['Year'])})."
)

Highlighted Asian country: India. Latest available total emissions: 2,661.1 Mt (2022).


---

## Submission checklist

- Keep `co2_emissions.csv` in the project's `data/` folder, or alongside the notebook.
- Run the notebook from the first cell to the last cell so both charts and printed values appear.
- Save the completed notebook as `week03/lecture03_exercise.ipynb` in your GitHub repository.
